In [1]:
from pathlib import Path

# Find repo root
REPO_ROOT = Path.cwd().parent
print(f"Repo root: {REPO_ROOT}")

REPORT_ROOT = REPO_ROOT / "report"

FIGSIZE = (20,18)
DPI = 100
GENERATE_PLOTS = False

Repo root: /Users/jedrek/Documents/Studium Volkswirschaftslehre/4. Semester/DEDA Project/DEDA_LLM_Spatial_Hotelling


In [2]:
import pandas as pd
import geopandas as gpd
import numpy as np
from pathlib import Path
import sys
import json
from shapely.geometry import shape

# Add both the repo root and src so imports work for scripts/ and hotelling/
sys.path.insert(0, str(Path.cwd().parent))
sys.path.insert(0, str(Path.cwd().parent / 'src'))

from hotelling.spatial.boundaries import load_boundary

PATH_RAW = REPO_ROOT / Path('data/raw')
PATH_PROCESSED = REPO_ROOT / Path('data/processed')

# Midpoint table (center coordinates)
zensus = gpd.read_parquet(PATH_RAW / 'zensus2022_grid.parquet')
zensus_filtered = gpd.read_parquet(PATH_RAW / 'zensus2022_grid_filtered.parquet')
lor = gpd.read_parquet(PATH_PROCESSED / 'lor.parquet')

# CRITICAL FIX: berlin.geojson has EPSG:3035 coordinates but geopandas auto-detects as EPSG:4326
# We must force the correct CRS instead of transforming from the wrong one
with open(PATH_RAW / 'city_boundary_Berlin.geojson', 'r') as f:
    berlin_json = json.load(f)
berlin = gpd.GeoDataFrame([1], geometry=[shape(berlin_json['geometry'])], crs='EPSG:3035')

boundary = load_boundary(PATH_RAW / 'relation_boundary_14983.geojson')

from hotelling.spatial.census import build_grid_polygons

grid = gpd.read_parquet(PATH_PROCESSED / 'pop_grid.parquet')

# Pop grid was saved with point geometry (midpoints). Convert to 100m square polygons.
grid = build_grid_polygons(grid)
#grid['geometry'] = grid.apply(lambda row: row.geometry.buffer(50, cap_style='square'), axis=1)
grid['index'] = grid.index
print(f"Grid: {len(grid)} cells as square polygons")

Grid: 16170 cells as square polygons


In [3]:
grid_malls = gpd.read_parquet(PATH_PROCESSED / 'grid_malls.parquet').to_crs(grid.crs)
grid_with_stations = gpd.read_parquet(PATH_PROCESSED / 'grid_with_stations.parquet').to_crs(grid.crs)
travel_times = pd.read_parquet(PATH_PROCESSED / 'travel_times.parquet')
employment_clusters = gpd.read_parquet(PATH_PROCESSED / 'employment_clusters.parquet').to_crs(grid.crs)
supermarkets = gpd.read_parquet(PATH_PROCESSED / 'supermarkets.parquet').to_crs(grid.crs)

brw = gpd.read_file(PATH_RAW / 'brw_2025.gpkg').to_crs(grid.crs)

In [4]:
'''def gitter_id(row):
    if row['GITTER_ID_100m'] is not None:
        return row['GITTER_ID_100m']
    else: 
        return str(f"CRS3035RES100mN{row['y_mp_100m']}E{row['x_mp_100m']}")
    
grid['cell_id'] = grid.apply(gitter_id, axis=1)

grid_travel_times = grid.merge(
    travel_times[["from_id", "to_id", "travel_time"]],
    left_on="cell_id",
    right_on="from_id",
    how="left"
)'''

'def gitter_id(row):\n    if row[\'GITTER_ID_100m\'] is not None:\n        return row[\'GITTER_ID_100m\']\n    else: \n        return str(f"CRS3035RES100mN{row[\'y_mp_100m\']}E{row[\'x_mp_100m\']}")\n\ngrid[\'cell_id\'] = grid.apply(gitter_id, axis=1)\n\ngrid_travel_times = grid.merge(\n    travel_times[["from_id", "to_id", "travel_time"]],\n    left_on="cell_id",\n    right_on="from_id",\n    how="left"\n)'

In [5]:
demand_grid = grid.copy()

def name_grid(row):
    #return f"CRS3035RES100mN{str(row['y_mp_100m'])[:5] + '00'}E{str(row['x_mp_100m'])[:5] + '00'}"
    #return f"CRS3035RES100mN{row['y_mp_100m']}E{row['x_mp_100m']}" or row['GITTER_ID_100m']
    if row['GITTER_ID_100m'] is not None:
        return row['GITTER_ID_100m']
    else: 
        return str(f"CRS3035RES100mN{row['y_mp_100m']}E{row['x_mp_100m']}")

demand_grid['GITTER_ID_100m'] = demand_grid.apply(name_grid, axis=1)
grid_malls['GITTER_ID_100m'] = grid_malls.apply(name_grid, axis=1)
grid_with_stations['GITTER_ID_100m'] = grid_with_stations.apply(name_grid, axis=1)

grid_with_stations['has_station'] = grid_with_stations['station_class'].notna()
grid_malls_true = grid_malls[grid_malls['has_mall']]
grid_stations_true = grid_with_stations[grid_with_stations['has_station']]

In [6]:
demand_grid['has_mall'] = False
demand_grid['has_station'] = False

demand_grid.loc[demand_grid['GITTER_ID_100m'].isin(grid_malls_true['GITTER_ID_100m']), 'has_mall'] = True
demand_grid.loc[demand_grid['GITTER_ID_100m'].isin(grid_stations_true['GITTER_ID_100m']), 'has_station'] = True

# Populate station_class and matched_db_station for cells with stations
station_info = grid_with_stations.set_index('GITTER_ID_100m')[['station_class', 'matched_db_station']]
demand_grid = demand_grid.join(station_info, on='GITTER_ID_100m')
demand_grid = demand_grid.sjoin(employment_clusters, how='left', predicate='intersects')

In [7]:
demand_grid['travel_times'] = None
# Precompute lookup once instead of filtering per row
travel_times['to_id'] = travel_times['to_id'].apply(lambda x: str(x))
travel_time_lookup = (
    travel_times.groupby('from_id')
    .apply(lambda df: df.set_index('to_id')['travel_time'].to_dict())
)

demand_grid['travel_times'] = demand_grid['GITTER_ID_100m'].map(travel_time_lookup)


/var/folders/y8/4_9g68pj7k136q2yypgp5ysc0000gn/T/ipykernel_22646/2396783161.py:6: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda df: df.set_index('to_id')['travel_time'].to_dict())


In [8]:
demand_grid

,x_mp_100m,y_mp_100m,geometry,GITTER_ID_100m,Einwohner,index,has_mall,has_station,station_class,matched_db_station,index_right,centroid_x,centroid_y,cluster_id,sum,n_cells,area,travel_times
0,4543150,3266250,"POLYGON ((4543200 3266200, 4543200 3266300, 45...",CRS3035RES100mN3266250E4543150,0,0,False,False,NaN,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"{'0': 57.0, '1': 32.0, '2': 58.0, '3': nan, '4..."
1,4543250,3266250,"POLYGON ((4543300 3266200, 4543300 3266300, 45...",CRS3035RES100mN3266250E4543250,0,1,False,False,NaN,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"{'0': 56.0, '1': 31.0, '2': 57.0, '3': nan, '4..."
2,4543350,3266250,"POLYGON ((4543400 3266200, 4543400 3266300, 45...",CRS3035RES100mN3266250E4543350,0,2,False,False,NaN,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"{'0': 54.0, '1': 29.0, '2': 55.0, '3': nan, '4..."
3,4543450,3266250,"POLYGON ((4543500 3266200, 4543500 3266300, 45...",CRS3035RES100mN3266250E4543450,0,3,False,False,NaN,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"{'0': 53.0, '1': 28.0, '2': 54.0, '3': nan, '4..."
4,4543550,3266250,"POLYGON ((4543600 3266200, 4543600 3266300, 45...",CRS3035RES100mN3266200E4543500,16,4,False,False,NaN,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"{'0': 52.0, '1': 27.0, '2': 53.0, '3': nan, '4..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16165,4557350,3277150,"POLYGON ((4557400 3277100, 4557400 3277200, 45...",CRS3035RES100mN3277150E4557350,0,16165,False,False,NaN,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"{'0': 47.0, '1': nan, '2': 56.0, '3': 21.0, '4..."
16166,4557450,3277150,"POLYGON ((4557500 3277100, 4557500 3277200, 45...",CRS3035RES100mN3277150E4557450,0,16166,False,False,NaN,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"{'0': 48.0, '1': nan, '2': 57.0, '3': 22.0, '4..."
16167,4557550,3277150,"POLYGON ((4557600 3277100, 4557600 3277200, 45...",CRS3035RES100mN3277150E4557550,0,16167,False,False,NaN,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"{'0': 49.0, '1': nan, '2': 58.0, '3': 23.0, '4..."
16168,4557650,3277150,"POLYGON ((4557700 3277100, 4557700 3277200, 45...",CRS3035RES100mN3277150E4557650,0,16168,False,False,NaN,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"{'0': 50.0, '1': nan, '2': 59.0, '3': 24.0, '4..."


In [9]:
from hotelling.spatial.city_data import download_index_data
download_index_data()

mss = gpd.read_file(PATH_RAW / 'mss.gpkg')
esix = gpd.read_file(PATH_RAW / 'esix.gpkg')

In [10]:
mss = mss.to_crs(demand_grid.crs)
esix = esix.to_crs(demand_grid.crs)

mss = mss.rename(columns={'plr_id': 'plr_id_mss'})
esix = esix.rename(columns={'plr_id': 'plr_id_esix'})

demand_grid = demand_grid.drop(columns=['index_right'])

demand_grid = demand_grid.sjoin(mss, how='left', predicate='intersects')
demand_grid = demand_grid.drop(columns=['index_right'])
demand_grid = demand_grid.sjoin(esix, how='left', predicate='intersects')
demand_grid = demand_grid.drop(columns=['index_right'])

In [11]:
# Map esix_wert to interval [0,1]
demand_grid['esix_normalized'] = (demand_grid['esix_wert'] - demand_grid['esix_wert'].min()) / (demand_grid['esix_wert'].max() - demand_grid['esix_wert'].min())

demand_grid['si_n'] = demand_grid['si_n'].replace({-9999: np.nan})
demand_grid['si_normalized'] = demand_grid['si_n'].replace({1.0: demand_grid['esix_normalized'].quantile([0.2, 0.4, 0.6, 0.8]).values[3],
                                                            2.0: demand_grid['esix_normalized'].quantile([0.2, 0.4, 0.6, 0.8]).values[2],
                                                            3.0: demand_grid['esix_normalized'].quantile([0.2, 0.4, 0.6, 0.8]).values[1],
                                                            4.0: demand_grid['esix_normalized'].quantile([0.2, 0.4, 0.6, 0.8]).values[0]})

In [12]:
demand_grid.to_parquet(PATH_PROCESSED / 'demand_grid.parquet')

In [13]:
supermarkets = supermarkets.sjoin(brw[['bezirk', 'brw', 'nutzung', 'geometry']], how='left', predicate='intersects')
supermarkets['id'] = supermarkets.index
supermarkets.to_parquet(PATH_PROCESSED / 'supermarkets_full.parquet')